# Memory X-Ray v0.2: full experiment (6 frozen scenarios)

**Run this only after approval.** The runner refuses to start unless all of the following hold:

1. **G1:** every frozen scenario and criteria file matches `scenarios/FREEZE.json`;
2. **G2:** the two label files in `records/labels/` are complete, rubric-valid and from distinct labelers, and their signals match the frozen reference answers. In this study they come from independent **AI-assisted** rubric validation (GPT-6 and Gemini; amendment 4), not human annotation;
3. **G3:** you set `APPROVED_BY` below;
4. **G4:** the pinned model commit `f39ac1d2…` loads and verifies.

Runtime: T4 GPU, with float16 storage and float32 compute. Expect roughly 40–70 minutes plus the model download. Keep the tab open.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code
Upload `memory_xray_v02.zip` (package version 0.2.4, which already contains the label files). Any older copy of the folder is moved aside, not deleted.

In [ ]:
import os, time, zipfile
from google.colab import files
if os.path.isdir("memory_xray_v02"):
    os.rename("memory_xray_v02", f"memory_xray_v02_old_{int(time.time())}")
uploaded = files.upload()
zipfile.ZipFile(next(iter(uploaded))).extractall(".")
%cd memory_xray_v02
import mxr
assert mxr.__version__ == "0.2.4", f"stale package: {mxr.__version__}"
print("package", mxr.__version__)

## 2. Install, test, and check gates G1 and G2

In [ ]:
!pip install -q "transformers>=4.41.2" "accelerate>=0.31" pytest
!python -m pytest -q
!python validate_scenarios.py --verify-freeze
!python check_labels.py

## 3. Approval (G3)
Type the approver's name. Leave it empty to stop here.

In [ ]:
APPROVED_BY = ""   # e.g. "Farnoosh Memari"
assert APPROVED_BY.strip(), "Not approved: set APPROVED_BY to run the full experiment."

## 4. Run the full experiment

In [ ]:
!python run_experiment.py --model microsoft/Phi-3-mini-4k-instruct --revision f39ac1d28e925b323eae81227eaba4464caced4e --storage-dtype float16 --compute-dtype float32 --confirm-full-run --approved-by "$APPROVED_BY"

## 5. View the summary and download the results

In [ ]:
import glob, shutil
from IPython.display import Markdown, display
run_dir = sorted(d for d in glob.glob("runs/full_*") if os.path.isdir(d))[-1]
print(run_dir)
display(Markdown(open(os.path.join(run_dir, "SUMMARY.md")).read()))
archive = shutil.make_archive(run_dir, "zip", run_dir)
files.download(archive)